In [1]:
import pandas as pd
import json
import glob

In [2]:
#!/usr/bin/env python3
"""Parse result directory names to extract model and parser/setting names."""

from pathlib import Path

# Known parsers/settings (suffixes after model name)
PARSERS = [
    "no_visual_cue_numeric",
    "no_visual_cue",
    "numeric",
    "default",
]


def parse_result_dir(dir_path: str | Path) -> tuple[str, str]:
    """
    Extract model name and parser name from a result directory path.
    Auto-detects model name by stripping known parser suffixes.

    Args:
        dir_path: Path like '../results/webarena_lite/claude_45_sonnet_numeric'

    Returns:
        Tuple of (model_name, parser_name)

    Raises:
        ValueError: If unable to parse the directory name
    """
    dir_name = Path(dir_path).name

    # Try to match known parser suffixes (longest first)
    for parser in PARSERS:
        suffix = f"_{parser}"
        if dir_name.endswith(suffix):
            model = dir_name[: -len(suffix)]
            if model:  # ensure model name is non-empty
                return (model, parser)

    raise ValueError(f"Unable to parse directory name: {dir_name}")



In [3]:
dirs = glob.glob("../results/webarena_lite/*") #+ glob.glob("../results/webarena_lite_bak/*")

In [4]:
dfs = {}
for model in dirs:
    print(model)
    if 'numeric' in model:
        continue
    data = []
    for file in glob.glob(f"{model}/*/result.json"):
        # print(file)
        with open(file, "r") as f:
            data.append(json.load(f))
    df = pd.DataFrame(data)
    dfs[model] = df
# del dfs['../results/webarena_lite/claude_45_fix_1']
# del dfs['../results/webarena_lite/claude_45_fix']


../results/webarena_lite/qwen3-4b-sft-128k-step22_default


../results/webarena_lite/qwen3-30b-a3b-rl-step99_default
../results/webarena_lite/claude_40_sonnet_default
../results/webarena_lite/claude_35_sonnet_no_visual_cue_numeric
../results/webarena_lite/qwen3-30b-a3b-sft-128k-step66_default
../results/webarena_lite/gpt_4o_default
../results/webarena_lite/qwen3-30b-a3b-rl-step19_default
../results/webarena_lite/claude_40_sonnet_numeric
../results/webarena_lite/claude_35_sonnet_no_visual_cue
../results/webarena_lite/qwen25_3b_default
../results/webarena_lite/o3_fix_prompt_default
../results/webarena_lite/qwen3-4b-rl-step19_default
../results/webarena_lite/qwen3-30b-a3b-rl-step139_default
../results/webarena_lite/qwen3-4b-rl-step79_default
../results/webarena_lite/qwen3-30b-a3b-rl-step159_default
../results/webarena_lite/claude_45_sonnet_no_visual_cue_numeric
../results/webarena_lite/claude_37_sonnet_no_visual_cue_numeric
../results/webarena_lite/claude_40_sonnet_vlm_no_visual_cue
../results/webarena_lite/llama31_8b_default
../results/webarena_l

In [5]:
result = []
for key, value in dfs.items():
    # print(key.split("/")[-1])
    if 'task_config' not in value:
        print(value)
    value['site'] = value['task_config'].apply(lambda x: x['sites'][0])
    value['score'] = value['result'].apply(lambda x: x['score'] if x else 0)
    # print(value.groupby('site')['score'].mean())
    # print(value.groupby('site').size())
    # print(value['score'].mean())
    model, setting = parse_result_dir(key)
    result.append({
        'model': model,
        'setting': setting,
        **value.groupby('site')['score'].mean(),
        'mean': value['score'].mean(),
        'num_tasks': len(value)
    })


In [6]:
import re

_STEP_RE = re.compile(r"^(.*?)-step(\d+)$")


def _model_sort_key(model: str) -> tuple[str, int]:
    """Split '<base>-step<N>' so sorting is by base name then numeric step.

    Non-step model names sort before any step variant with the same base.
    """
    m = _STEP_RE.match(model)
    if m:
        return (m.group(1), int(m.group(2)))
    return (model, -1)


df = pd.DataFrame(result)
# sort by (base model, numeric step, setting) so e.g. step19 < step99 < step119
df = df.sort_values(
    by=["model", "setting"],
    key=lambda col: col.map(_model_sort_key) if col.name == "model" else col,
)
df


,model,setting,gitlab,shopping,shopping_admin,mean,num_tasks
25,claude_35_sonnet,default,0.366667,0.266667,0.314286,0.309091,110
6,claude_35_sonnet,no_visual_cue,0.066667,0.111111,0.171429,0.118182,110
42,claude_37_sonnet,default,0.500000,0.311111,0.371429,0.381818,110
32,claude_37_sonnet,no_visual_cue,0.166667,0.155556,0.171429,0.163636,110
2,claude_40_sonnet,default,0.500000,0.422222,0.485714,0.463636,110
29,claude_40_sonnet,no_visual_cue,0.333333,0.311111,0.342857,0.327273,110
30,claude_40_sonnet_vlm,default,0.433333,0.444444,0.485714,0.454545,110
13,claude_40_sonnet_vlm,no_visual_cue,0.366667,0.377778,0.342857,0.363636,110
16,claude_45_sonnet,default,0.500000,0.400000,0.628571,0.500000,110
21,claude_45_sonnet,no_visual_cue,0.533333,0.377778,0.342857,0.409091,110


In [7]:
df[df['setting'] == 'default'].to_csv("result_default.csv", index=False)

In [8]:
import re
import matplotlib.pyplot as plt

_STEP_RE = re.compile(r"^(.*?)-step(\d+)$")

# Each RL job pairs with an SFT checkpoint that defines its step-0 anchor.
# (rl_base_prefix, sft_step0_model_name)
RL_JOBS = [
    ("qwen3-30b-a3b-rl",   "qwen3-30b-a3b-sft-128k-step66"),
    ("qwen3-4b-rl",        "qwen3-4b-sft-128k-step66"),
]

default = df[df["setting"] == "default"].set_index("model")

fig, ax = plt.subplots(figsize=(8, 5))
for base, sft_step0 in RL_JOBS:
    points = []
    # step 0 anchor from paired SFT checkpoint
    if sft_step0 in default.index:
        points.append((0, default.loc[sft_step0, "mean"]))
    # all step checkpoints of this RL job
    for model in default.index:
        m = _STEP_RE.match(model)
        if m and m.group(1) == base:
            points.append((int(m.group(2)), default.loc[model, "mean"]))
    points.sort(key=lambda p: p[0])
    xs, ys = zip(*points)
    ax.plot(xs, ys, marker="o", label=base)
    # annotate step-0 so the SFT anchor is visible
    ax.annotate(f"{ys[0]:.3f}", (xs[0], ys[0]),
                textcoords="offset points", xytext=(5, 5), fontsize=8)

ax.set_xlabel("RL step")
ax.set_ylabel("mean success rate")
ax.set_title("RL training curve (step 0 = paired SFT-128k-step66)")
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()


ModuleNotFoundError: No module named 'matplotlib'

In [15]:
errors = dfs['../results/webarena_lite/qwen25_3b_default']['result'].apply(lambda x: x['error'] if 'error' in x else None).apply(lambda x: None if x is not None and "ValidationException" in x else x)
errors[errors.notna()]

1      Error code: 400 - {'object': 'error', 'message...
3      Error code: 400 - {'object': 'error', 'message...
9      Error code: 400 - {'object': 'error', 'message...
12     Error code: 400 - {'object': 'error', 'message...
13     Error code: 400 - {'object': 'error', 'message...
16     Error code: 400 - {'object': 'error', 'message...
21     Error code: 400 - {'object': 'error', 'message...
22     Error code: 400 - {'object': 'error', 'message...
30                                                      
31     502 Server Error: Bad Gateway for url: http://...
33     Error code: 400 - {'object': 'error', 'message...
35     Error code: 400 - {'object': 'error', 'message...
48                Account vinta not found on GitLab page
53     Incus server not available at http://127.0.0.1...
54     Incus server not available at http://127.0.0.1...
56     Error code: 400 - {'object': 'error', 'message...
58     Error code: 400 - {'object': 'error', 'message...
60     Error code: 400 - {'obje

In [63]:
errors[errors.notna()]

11    502 Server Error: Bad Gateway for url: http://...
23    Incus server not available at http://127.0.0.1...
24    Incus server not available at http://127.0.0.1...
27    Error code: 400 - {'object': 'error', 'message...
35    Unknown required_contents: dict_keys(['fuzzy_m...
48    Error code: 400 - {'object': 'error', 'message...
Name: result, dtype: object

In [40]:
errors = dfs['../results/webarena_lite_bak/claude_45']['result'].apply(lambda x: x['error'] if 'error' in x else None).apply(lambda x: None if x is not None and "ValidationException" in x else x)
errors[errors.notna()].iloc[1]

"Page.evaluate: TypeError: Cannot read properties of null (reading 'lastChild')\n    at eval (eval at evaluate (:291:30), <anonymous>:1:46)\n    at UtilityScript.evaluate (<anonymous>:298:18)\n    at UtilityScript.<anonymous> (<anonymous>:1:44)"

In [ ]:
errors = dfs['../results/webarena_lite/claude_45_sonnet_default']['result'].apply(lambda x: x['error'] if 'error' in x else None).apply(lambda x: None if x is not None and "ValidationException" in x else x)
dfs['../results/webarena_lite/claude_45_sonnet_default'][errors.notna()]

,task_id,task_config,result,execution_time,trace_summary,site,score
30,81,"{'sites': ['gitlab'], 'task_id': 81, 'require_...","{'success': False, 'score': 0.0, 'answer': '',...","{'start': '2026-01-01T06:39:58.767808', 'end':...","{'total_steps': 0, 'final_score': 0.0, 'succes...",gitlab,0.0
31,104,"{'sites': ['shopping'], 'task_id': 104, 'requi...","{'success': False, 'score': 0.0, 'answer': '',...","{'start': '2026-01-01T05:50:48.430145', 'end':...","{'total_steps': 0, 'final_score': 0.0, 'succes...",shopping,0.0
48,112,"{'sites': ['gitlab'], 'task_id': 112, 'require...","{'success': False, 'score': 0.0, 'answer': '',...","{'start': '2026-01-01T06:31:49.097901', 'end':...","{'total_steps': 0, 'final_score': 0.0, 'succes...",gitlab,0.0
75,123,"{'sites': ['shopping'], 'task_id': 123, 'requi...","{'success': False, 'score': 0.0, 'answer': 'Er...","{'start': '2026-01-01T06:39:40.975895', 'end':...","{'total_steps': 1, 'final_score': 0.0, 'succes...",shopping,0.0
78,142,"{'sites': ['shopping'], 'task_id': 142, 'requi...","{'success': False, 'score': 0.0, 'answer': 'Er...","{'start': '2026-01-01T05:48:29.038148', 'end':...","{'total_steps': 1, 'final_score': 0.0, 'succes...",shopping,0.0
90,117,"{'sites': ['shopping'], 'task_id': 117, 'requi...","{'success': False, 'score': 0.0, 'answer': '',...","{'start': '2026-01-01T06:25:16.131338', 'end':...","{'total_steps': 0, 'final_score': 0.0, 'succes...",shopping,0.0
91,118,"{'sites': ['shopping'], 'task_id': 118, 'requi...","{'success': False, 'score': 0.0, 'answer': '',...","{'start': '2026-01-01T06:36:54.104689', 'end':...","{'total_steps': 0, 'final_score': 0.0, 'succes...",shopping,0.0
103,130,"{'sites': ['gitlab'], 'task_id': 130, 'require...","{'success': False, 'score': 0.0, 'answer': '',...","{'start': '2026-01-01T06:30:38.662510', 'end':...","{'total_steps': 0, 'final_score': 0.0, 'succes...",gitlab,0.0
107,143,"{'sites': ['gitlab'], 'task_id': 143, 'require...","{'success': False, 'score': 0.0, 'answer': '',...","{'start': '2026-01-01T06:25:44.802579', 'end':...","{'total_steps': 0, 'final_score': 0.0, 'succes...",gitlab,0.0


dict_keys(['../results/webarena_lite/claude_45_sonnet_numeric', '../results/webarena_lite/claude_37_sonnet_numeric', '../results/webarena_lite/claude_35_sonnet_no_visual_cue_numeric', '../results/webarena_lite/claude_45_sonnet_default', '../results/webarena_lite/claude_37_sonnet_no_visual_cue_numeric', '../results/webarena_lite/claude_45_sonnet_no_visual_cue_numeric', '../results/webarena_lite/claude_35_sonnet_default', '../results/webarena_lite/claude_40_sonnet_no_visual_cue', '../results/webarena_lite/claude_45_sonnet_no_visual_cue', '../results/webarena_lite/claude_35_sonnet_no_visual_cue', '../results/webarena_lite/claude_40_sonnet_numeric', '../results/webarena_lite/claude_37_sonnet_no_visual_cue', '../results/webarena_lite/claude_40_sonnet_no_visual_cue_numeric', '../results/webarena_lite/claude_37_sonnet_default', '../results/webarena_lite/claude_40_sonnet_default', '../results/webarena_lite/claude_35_sonnet_numeric', '../results/webarena_lite_bak/claude_45_fix', '../results/web